# Quickstart

# Working with data

PyTorch has two [🔗 primitives to work with
data](https://pytorch.org/docs/stable/data.html):
1. ***[🔗 torch.utils.data.DataLoader](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader)***
2. ***[🔗 torch.utils.data.Dataset](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.Dataset)***

***Dataset*** stores the samples and their corresponding labels, and ***DataLoader***
wraps an iterable around the ***Dataset***.


In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

# Local
from data import data

PyTorch offers ***domain-specific libraries*** such as:
* [🔗 TorchText](https://pytorch.org/text/stable/index.html).
* [🔗 TorchVision](https://pytorch.org/vision/stable/index.html).
* [🔗 TorchAudio](https://pytorch.org/audio/stable/index.html).

All of which include ***datasets***.
For this notebook, we will be using a ***TorchVision dataset***.

The ***torchvision.datasets*** module contains ***Dataset*** objects for many real-world vision data like ***CIFAR***,
***COCO***
[[🔗 full list here](https://pytorch.org/vision/stable/datasets.html)].
In this notebook, we use the ***FashionMNIST dataset***.

Every ***TorchVision Dataset*** includes two arguments:
1. ***transform*** and
2. ***target_transform*** \- to modify the samples and labels, respectively.


In [2]:
# Path of the folder where the dataset will be stored.
training_data_location = data.get_dataset_path("Fashion-mnist", "raw", "train", 1)
test_data_location = data.get_dataset_path("Fashion-mnist", "raw", "test", 1)

In [3]:
# Download training data from open datasets.
training_data = datasets.FashionMNIST(
    root = training_data_location,
    train = True,
    download = True,
    transform = ToTensor(),
)

# Download test data from open datasets.
test_data = datasets.FashionMNIST(
    root = test_data_location,
    train = False,
    download = True,
    transform = ToTensor(),
)

We pass the `Dataset` as an argument to `DataLoader`. This wraps an
iterable over our dataset, and supports automatic batching, sampling,
shuffling and multiprocess data loading. Here we define a batch size of
64, i.e. each element in the dataloader iterable will return a batch of
64 features and labels.


In [4]:
batch_size = 64

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


Read more about [loading data in PyTorch](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html).


------------------------------------------------------------------------


Creating Models
===============

To define a neural network in PyTorch, we create a class that inherits
from
[nn.Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html).
We define the layers of the network in the `__init__` function and
specify how data will pass through the network in the `forward`
function. To accelerate operations in the neural network, we move it to
the
[accelerator](https://pytorch.org/docs/stable/torch.html#accelerators)
such as CUDA, MPS, MTIA, or XPU. If the current accelerator is
available, we will use it. Otherwise, we use the CPU.


In [5]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using {device} device")

Using mps device


In [6]:
# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Read more about [building neural networks in
PyTorch](buildmodel_tutorial.html).


------------------------------------------------------------------------


Optimizing the Model Parameters
===============================

To train a model, we need a [loss
function](https://pytorch.org/docs/stable/nn.html#loss-functions) and an
[optimizer](https://pytorch.org/docs/stable/optim.html).


In [7]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In a single training loop, the model makes predictions on the training
dataset (fed to it in batches), and backpropagates the prediction error
to adjust the model\'s parameters.


In [8]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

We also check the model\'s performance against the test dataset to
ensure it is learning.


In [9]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

The training process is conducted over several iterations (*epochs*).
During each epoch, the model learns parameters to make better
predictions. We print the model\'s accuracy and loss at each epoch;
we\'d like to see the accuracy increase and the loss decrease with every
epoch.


In [10]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.298994  [   64/60000]
loss: 2.291126  [ 6464/60000]
loss: 2.268356  [12864/60000]
loss: 2.261944  [19264/60000]
loss: 2.245496  [25664/60000]
loss: 2.215442  [32064/60000]
loss: 2.230899  [38464/60000]
loss: 2.195010  [44864/60000]
loss: 2.176293  [51264/60000]
loss: 2.156047  [57664/60000]
Test Error: 
 Accuracy: 38.8%, Avg loss: 2.147689 

Epoch 2
-------------------------------
loss: 2.152648  [   64/60000]
loss: 2.151836  [ 6464/60000]
loss: 2.086367  [12864/60000]
loss: 2.104055  [19264/60000]
loss: 2.067434  [25664/60000]
loss: 1.994329  [32064/60000]
loss: 2.036355  [38464/60000]
loss: 1.954085  [44864/60000]
loss: 1.942191  [51264/60000]
loss: 1.891502  [57664/60000]
Test Error: 
 Accuracy: 57.0%, Avg loss: 1.882786 

Epoch 3
-------------------------------
loss: 1.907427  [   64/60000]
loss: 1.889765  [ 6464/60000]
loss: 1.767884  [12864/60000]
loss: 1.809844  [19264/60000]
loss: 1.721650  [25664/60000]
loss: 1.660437  [32064/600

Read more about [Training your model](optimization_tutorial.html).


------------------------------------------------------------------------


Saving Models
=============

A common way to save a model is to serialize the internal state
dictionary (containing the model parameters).


In [13]:
model_path = data.get_dataset_path("Fashion-mnist", "processed", "model", 1)
model_path

'/../../../../../../Volumes/Workstation/Datasets/PyTorch/fashion_mnist/models/model.pth'

In [14]:
torch.save(model.state_dict(), model_path)
print("Saved PyTorch Model State to model.pth")

Saved PyTorch Model State to model.pth


Loading Models
==============

The process for loading a model includes re-creating the model structure
and loading the state dictionary into it.


In [15]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load(model_path, weights_only=True))

<All keys matched successfully>

This model can now be used to make predictions.


In [16]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
x, y = test_data[0][0], test_data[0][1]
with torch.no_grad():
    x = x.to(device)
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

Predicted: "Ankle boot", Actual: "Ankle boot"


Read more about [Saving & Loading your
model](saveloadrun_tutorial.html).
